<a href="https://colab.research.google.com/github/usharani-192312304/ITA0605-ML/blob/main/Exp16_Compare_Classification_Algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 16: Comparing Classification Algorithms
**Course:** ITA06 - Machine Learning (Lab)

## Aim
Compare different types of classification algorithms and evaluate their performance.

## Background
No single classifier wins everywhere. We compare eight families on the same data with the same preprocessing and the same cross-validation folds: **Logistic Regression** (linear), **K-NN** (instance-based), **Naive Bayes** (probabilistic), **Decision Tree**, **Random Forest** and **Gradient Boosting** (tree-based, single vs ensemble), **SVM** (kernel / margin-based) and a **Neural Network (MLP)**. Metrics: accuracy, precision, recall, F1, ROC-AUC and training time.

## Pipeline followed in this notebook
1. Step 1: Data Collection
2. Step 2: Data Loading
3. Step 3: Data Preprocessing
4. Step 4: Exploratory Data Analysis (EDA)
5. Step 5: Model Training
6. Step 6: Model Evaluation
7. Step 7: Prediction on New Data

> **How to run:** Open in Google Colab -> *Runtime -> Run all*. No files need to be uploaded; the data is created or loaded inside the notebook.

## Setup: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)
print("Libraries imported successfully")

## Step 1: Data Collection
We use the **Breast Cancer Wisconsin (Diagnostic)** dataset from scikit-learn: 569 tumours, 30 numeric measurements, label = malignant / benign.

In [ ]:
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer(as_frame=True)
bc.frame.to_csv("breast_cancer.csv", index=False)
print("Saved breast_cancer.csv", bc.frame.shape)

## Step 2: Data Loading

In [ ]:
df = pd.read_csv("breast_cancer.csv")
df.head()

## Step 3: Data Preprocessing
Stratified train/test split. Scaling is put **inside each model's pipeline** (scale-sensitive models get a `StandardScaler`) so that cross-validation never leaks information from validation folds.

In [ ]:
from sklearn.model_selection import train_test_split

print("Missing values:", df.isnull().sum().sum(), "| Duplicates:", df.duplicated().sum())
X = df.drop(columns="target")
y = df["target"]                       # 0 = malignant, 1 = benign
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print("Train:", X_train.shape, "Test:", X_test.shape)

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
print(df["target"].map({0: "malignant", 1: "benign"}).value_counts())
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
tmp = df.assign(diagnosis=df["target"].map({0: "malignant", 1: "benign"}))
sns.countplot(data=tmp, x="diagnosis", ax=axes[0]); axes[0].set_title("Class balance")
sns.histplot(data=tmp, x="worst radius", hue="diagnosis", kde=True, ax=axes[1]); axes[1].set_title("worst radius")
sns.scatterplot(data=tmp, x="mean texture", y="mean concave points", hue="diagnosis", ax=axes[2]); axes[2].set_title("Two informative features")
plt.tight_layout(); plt.show()

corr_with_target = df.corr()["target"].drop("target").sort_values()
corr_with_target.plot(kind="barh", figsize=(7, 8), color="slateblue", title="Correlation of each feature with 'benign'"); plt.show()

## Step 5: Model Training
Define the eight classifiers and train them on the training set, timing the fit.

In [ ]:
import time
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "K-NN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
    "Neural Net (MLP)": make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=2000, random_state=42)),
}
fit_time = {}
for name, m in models.items():
    t0 = time.time()
    m.fit(X_train, y_train)
    fit_time[name] = time.time() - t0
print({k: f"{v:.3f}s" for k, v in fit_time.items()})

## Step 6: Model Evaluation
Two views: **(a)** 10-fold stratified cross-validation on the training set (robust, with spread); **(b)** performance on the held-out test set. Then ROC curves and confusion matrices side by side.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_rows = []
for name, m in models.items():
    r = cross_validate(m, X_train, y_train, cv=skf, scoring=["accuracy", "f1", "roc_auc"], n_jobs=-1)
    cv_rows.append({"Model": name, "CV Accuracy": r["test_accuracy"].mean(), "CV Acc Std": r["test_accuracy"].std(),
                    "CV F1": r["test_f1"].mean(), "CV ROC-AUC": r["test_roc_auc"].mean()})
cv_df = pd.DataFrame(cv_rows).set_index("Model").round(4).sort_values("CV Accuracy", ascending=False)
print(cv_df)

cv_df["CV Accuracy"].sort_values().plot(kind="barh", xerr=cv_df.loc[cv_df["CV Accuracy"].sort_values().index, "CV Acc Std"],
                                         color="steelblue", figsize=(8, 5), xlim=(0.88, 1.0))
plt.title("10-fold CV accuracy (error bars = std)"); plt.xlabel("Accuracy"); plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix

test_rows, roc_data, preds = [], {}, {}
for name, m in models.items():
    p = m.predict(X_test); prob = m.predict_proba(X_test)[:, 1]
    preds[name] = p; roc_data[name] = roc_curve(y_test, prob)
    test_rows.append({"Model": name, "Accuracy": accuracy_score(y_test, p), "Precision": precision_score(y_test, p),
                      "Recall": recall_score(y_test, p), "F1": f1_score(y_test, p), "ROC-AUC": roc_auc_score(y_test, prob),
                      "Fit time (s)": fit_time[name]})
test_df = pd.DataFrame(test_rows).set_index("Model").round(4).sort_values(["F1", "ROC-AUC"], ascending=False)
print(test_df)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, (fpr, tpr, _) in roc_data.items():
    ax.plot(fpr, tpr, label=f"{name} (AUC={test_df.loc[name, 'ROC-AUC']:.3f})")
ax.plot([0, 1], [0, 1], "k--"); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC curves - all models"); ax.legend(loc="lower right", fontsize=8); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, (name, p) in zip(axes.ravel(), preds.items()):
    sns.heatmap(confusion_matrix(y_test, p), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["malig.", "benign"], yticklabels=["malig.", "benign"])
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()

plt.scatter(test_df["Fit time (s)"], test_df["Accuracy"], s=80)
for name, row in test_df.iterrows():
    plt.annotate(name, (row["Fit time (s)"], row["Accuracy"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.xscale("log"); plt.xlabel("Fit time (s, log)"); plt.ylabel("Test accuracy"); plt.title("Accuracy vs training cost"); plt.show()

## Step 7: Prediction on New Data
Use the best model to classify new samples, and show every model's vote.

In [ ]:
best_name = test_df.index[0]
print("Best model on the test set (by F1, then ROC-AUC):", best_name)
sample = X_test.iloc[[0, 1, 2]]
votes = pd.DataFrame({name: m.predict(sample) for name, m in models.items()}, index=["sample 1", "sample 2", "sample 3"])
votes = votes.replace({0: "malignant", 1: "benign"})
votes["actual"] = y_test.iloc[:3].map({0: "malignant", 1: "benign"}).values
votes.T

## Conclusion
On this dataset every algorithm scores roughly 92-98% accuracy. In cross-validation Logistic Regression, the MLP and the RBF-SVM (scaled inputs) are at the top, Random Forest, K-NN and Gradient Boosting follow, and Naive Bayes and the single Decision Tree are lowest. Differences of 1-2% are within cross-validation noise (the standard deviations are 1-4%), so the final choice should also weigh interpretability, speed and recall for the *malignant* class (a missed cancer is costlier than a false alarm). 'Best' always depends on the data and the cost of errors.